# `higgstts` with VoiceHub

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadirnar/voicehub/blob/main/notebooks/models/higgstts.ipynb)

- Task: **Text to speech**
- Default checkpoint: [`bosonai/higgs-tts-2-3b-base`](https://huggingface.co/bosonai/higgs-tts-2-3b-base)

The registry check is safe to run without downloading weights. Inference is disabled by default.


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("voicehub") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "voicehub @ git+https://github.com/kadirnar/voicehub.git@main",
    ])


In [ ]:
from pathlib import Path

RUN_INFERENCE = False
MODEL_TYPE = 'higgstts'
CHECKPOINT = 'bosonai/higgs-tts-2-3b-base'
DEVICE = "cuda"

TEXT = "VoiceHub provides one clear and reproducible notebook for every registered Hub model."
REFERENCE_AUDIO = Path("reference.wav")
REFERENCE_TEXT = "This transcript must exactly match the authorized reference audio."
OUTPUT_FILE = Path("artifacts/higgstts.wav")
GENERATION_KWARGS = {
}


## Inspect registry support


In [ ]:
from voicehub import get_model_spec

model_spec = get_model_spec(MODEL_TYPE)
assert model_spec.task.value == 'text-to-speech'
assert model_spec.default_model_path == CHECKPOINT
print("task:", model_spec.task.value)
print("checkpoint:", model_spec.default_model_path)
print("capabilities:", ", ".join(model_spec.capabilities))
print("training:", model_spec.training.support.value)


## Run inference

Set `RUN_INFERENCE = True`. Models that clone or prompt a voice also need an authorized `reference.wav`; review `GENERATION_KWARGS` before running.


In [ ]:
if RUN_INFERENCE:
    from IPython.display import Audio, display

    from voicehub import AutoModelForTextToSpeech, TTSGenerationConfig

    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    model = AutoModelForTextToSpeech.from_pretrained(
        CHECKPOINT,
        model_type=MODEL_TYPE,
        device=DEVICE,
        lazy_load=True,
    )
    output = model.generate(
        TEXT,
        generation_config=TTSGenerationConfig(seed=42, output_file=OUTPUT_FILE),
        **GENERATION_KWARGS,
    )
    print(output.file_path, output.sample_rate, output.metadata)
    display(Audio(output.audio, rate=output.sample_rate))


## Next

See the [inference guide](https://kadirnar.github.io/voicehub/guides/inference/) and [model catalog](https://kadirnar.github.io/voicehub/models/) for the shared runtime contract and model-specific limitations.
